# HW 5 Task 2: Encoder-Decoder Seq2Seq With Scaled Dot-Product Attention

This notebook implements a small **encoder-decoder seq2seq model** from scratch using only `NumPy` and `pandas`, then integrates **scaled dot-product attention into the encoder architecture**.

Model choice:
- **Encoder**: simple RNN encoder
- **Attention integration in encoder**: scaled dot-product **self-attention** over encoder hidden states
- **Decoder**: simple RNN decoder
- **Decoder attention**: Luong-style dot-product attention over the encoder memory

This follows the Lecture 10 theme of encoder-decoder models plus attention, while using a clean NumPy-only forward pass.


## Architecture

For a source sequence `x_1, ..., x_T`:

1. The RNN encoder produces hidden states `h_1, ..., h_T`.
2. We apply scaled dot-product self-attention over those encoder states:

$$
A = \mathrm{softmax}\left(\frac{HH^T}{\sqrt{d}}\right)
$$

3. The attention-enhanced encoder memory is:

$$
\widetilde{H} = AH
$$

4. The decoder attends over `\widetilde{H}` while generating outputs.

This is a valid way to integrate attention **inside the encoder architecture**, since the encoder representations are refined before they are passed to the decoder.


In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)


## Utility Functions

In [ ]:
def softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def tanh(x):
    return np.tanh(x)


def scaled_dot_product_attention(query, key, value, mask=None):
    query = np.asarray(query, dtype=float)
    key = np.asarray(key, dtype=float)
    value = np.asarray(value, dtype=float)

    d_k = key.shape[-1]
    scores = query @ key.T
    scaled_scores = scores / np.sqrt(d_k)

    if mask is not None:
        scaled_scores = np.where(mask, scaled_scores, -1e9)

    attention_weights = softmax(scaled_scores, axis=-1)
    output = attention_weights @ value
    return output, attention_weights, scaled_scores


## Tiny Vocabulary and Toy Parallel Data

We use a minimal example with source and target tokens just to demonstrate the forward pass clearly.


In [ ]:
source_tokens = ["<bos>", "i", "like", "nlp", "<eos>"]
target_tokens = ["<bos>", "j", "aime", "nlp", "<eos>"]

src_vocab = {token: idx for idx, token in enumerate(sorted(set(source_tokens)))}
tgt_vocab = {token: idx for idx, token in enumerate(sorted(set(target_tokens)))}
tgt_id_to_token = {idx: token for token, idx in tgt_vocab.items()}

src_ids = np.array([src_vocab[token] for token in source_tokens])
tgt_ids = np.array([tgt_vocab[token] for token in target_tokens])

pd.DataFrame({
    "source_token": source_tokens,
    "source_id": src_ids,
    "target_token": target_tokens,
    "target_id": tgt_ids,
})


## Parameter Initialization

The model is intentionally small. We are not training it here; the goal is to show the seq2seq forward computation with encoder attention.


In [ ]:
embed_dim = 6
hidden_dim = 8

src_embedding = rng.normal(0, 0.5, size=(len(src_vocab), embed_dim))
tgt_embedding = rng.normal(0, 0.5, size=(len(tgt_vocab), embed_dim))

W_xh_enc = rng.normal(0, 0.4, size=(embed_dim, hidden_dim))
W_hh_enc = rng.normal(0, 0.4, size=(hidden_dim, hidden_dim))
b_enc = np.zeros(hidden_dim)

W_xh_dec = rng.normal(0, 0.4, size=(embed_dim, hidden_dim))
W_hh_dec = rng.normal(0, 0.4, size=(hidden_dim, hidden_dim))
W_ctx_dec = rng.normal(0, 0.4, size=(hidden_dim, hidden_dim))
b_dec = np.zeros(hidden_dim)

W_out = rng.normal(0, 0.4, size=(hidden_dim * 2, len(tgt_vocab)))
b_out = np.zeros(len(tgt_vocab))


## Encoder

We first run a vanilla RNN encoder, then refine its hidden states with scaled dot-product self-attention.


In [ ]:
def rnn_encoder_forward(token_ids, embedding_matrix, W_xh, W_hh, b):
    hidden_states = []
    h_t = np.zeros(W_hh.shape[0])

    for token_id in token_ids:
        x_t = embedding_matrix[token_id]
        h_t = tanh(x_t @ W_xh + h_t @ W_hh + b)
        hidden_states.append(h_t.copy())

    return np.vstack(hidden_states)


encoder_states = rnn_encoder_forward(src_ids, src_embedding, W_xh_enc, W_hh_enc, b_enc)
encoder_memory, encoder_self_attention, encoder_scores = scaled_dot_product_attention(
    encoder_states,
    encoder_states,
    encoder_states,
)

print("Raw encoder hidden states shape:", encoder_states.shape)
print("Attention-enhanced encoder memory shape:", encoder_memory.shape)


In [ ]:
encoder_state_df = pd.DataFrame(
    encoder_states,
    index=source_tokens,
    columns=[f"h_{i}" for i in range(hidden_dim)],
)

encoder_memory_df = pd.DataFrame(
    encoder_memory,
    index=source_tokens,
    columns=[f"m_{i}" for i in range(hidden_dim)],
)

encoder_attention_df = pd.DataFrame(
    encoder_self_attention,
    index=[f"query:{token}" for token in source_tokens],
    columns=[f"key:{token}" for token in source_tokens],
)

print("Encoder hidden states:")
display(encoder_state_df)

print("Encoder self-attention weights:")
display(encoder_attention_df)

print("Attention-enhanced encoder memory:")
display(encoder_memory_df)


## Decoder With Luong-Style Attention Over Encoder Memory

At each decoder step:
- update decoder hidden state with a simple RNN
- use that decoder state as the query
- attend over the **attention-enhanced encoder memory**
- combine decoder state and context to produce output logits


In [ ]:
def decoder_forward(target_ids, embedding_matrix, encoder_memory, W_xh, W_hh, W_ctx, b, W_out, b_out):
    h_t = np.zeros(W_hh.shape[0])
    contexts = []
    attn_weights_all = []
    logits_all = []

    for token_id in target_ids[:-1]:
        x_t = embedding_matrix[token_id]
        context, attn_weights, _ = scaled_dot_product_attention(
            h_t.reshape(1, -1),
            encoder_memory,
            encoder_memory,
        )
        context = context[0]
        h_t = tanh(x_t @ W_xh + h_t @ W_hh + context @ W_ctx + b)

        combined = np.concatenate([h_t, context])
        logits = combined @ W_out + b_out

        contexts.append(context.copy())
        attn_weights_all.append(attn_weights[0].copy())
        logits_all.append(logits.copy())

    return np.vstack(logits_all), np.vstack(attn_weights_all), np.vstack(contexts)


decoder_logits, decoder_attention, decoder_contexts = decoder_forward(
    tgt_ids,
    tgt_embedding,
    encoder_memory,
    W_xh_dec,
    W_hh_dec,
    W_ctx_dec,
    b_dec,
    W_out,
    b_out,
)

decoder_probabilities = softmax(decoder_logits, axis=1)
predicted_ids = decoder_probabilities.argmax(axis=1)
predicted_tokens = [tgt_id_to_token[idx] for idx in predicted_ids]

print("Decoder logits shape:", decoder_logits.shape)
print("Decoder attention shape:", decoder_attention.shape)
print("Predicted target tokens:", predicted_tokens)


In [ ]:
decoder_steps = [f"step_{i+1}" for i in range(len(target_tokens) - 1)]

decoder_attention_df = pd.DataFrame(
    decoder_attention,
    index=decoder_steps,
    columns=source_tokens,
)

decoder_context_df = pd.DataFrame(
    decoder_contexts,
    index=decoder_steps,
    columns=[f"c_{i}" for i in range(hidden_dim)],
)

decoder_probs_df = pd.DataFrame(
    decoder_probabilities,
    index=decoder_steps,
    columns=[tgt_id_to_token[i] for i in range(len(tgt_vocab))],
)

print("Decoder attention over encoder memory:")
display(decoder_attention_df)

print("Decoder context vectors:")
display(decoder_context_df)

print("Decoder output probabilities:")
display(decoder_probs_df)


## Why This Satisfies the Requirement

This is an **encoder-decoder seq2seq model** with attention integrated into the **encoder architecture**:

- The encoder first builds sequential hidden states with an RNN.
- Those hidden states are then refined by **scaled dot-product self-attention**.
- The decoder consumes this attention-enhanced encoder memory.

This is close in spirit to later self-attention-based encoders, while the decoder attention remains compatible with the classic seq2seq view from Bahdanau/Luong-style models.


In [ ]:
assert encoder_states.shape == encoder_memory.shape
assert np.allclose(encoder_self_attention.sum(axis=1), 1.0)
assert np.allclose(decoder_attention.sum(axis=1), 1.0)

summary = pd.DataFrame({
    "decoder_input": target_tokens[:-1],
    "predicted_token": predicted_tokens,
})

print("Checks passed: encoder and decoder attention rows sum to 1.")
display(summary)
